# W6: CRM

고객 주문 데이터를 RFM으로 분류하고 이탈고객용 Gemini 메시지를 생성합니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

import numpy as np
orders = pd.DataFrame({
    "고객ID": ["A", "B", "C", "D", "E", "F", "G", "H"],
    "최근구매일": ["2026-01-30", "2026-01-10", "2025-12-20", "2026-01-25", "2026-01-28", "2025-11-01", "2026-01-29", "2026-01-05"],
    "구매횟수": [9, 6, 2, 5, 11, 1, 8, 3],
    "총구매금액": [450000, 280000, 42000, 130000, 620000, 15000, 390000, 65000]
})
orders["최근구매일"] = pd.to_datetime(orders["최근구매일"])
latest = orders["최근구매일"].max()
orders["Recency"] = (latest - orders["최근구매일"]).dt.days

orders["RecencyScore"] = pd.qcut(orders["Recency"].rank(method="first"), 4, labels=[4,3,2,1]).astype(int)
orders["FrequencyScore"] = pd.qcut(orders["구매횟수"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
orders["MonetaryScore"] = pd.qcut(orders["총구매금액"].rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
orders["Score"] = orders["RecencyScore"] + orders["FrequencyScore"] + orders["MonetaryScore"]


def segment(score):
    if score >= 10:
        return "VIP"
    if score >= 7:
        return "일반"
    if score <= 4:
        return "이탈고객"
    return "재구매유도"

orders["세그먼트"] = orders["Score"].apply(segment)


def msg(row):
    prompt = f"고객ID:{row.고객ID}, 세그먼트:{row.세그먼트}. 이탈방지용 문자 2문장 생성"
    fallback = "오랜만이에요. 좋은 제안이 있어 다시 알려드리니 이번 주 쿠폰으로 확인해 주세요."
    return use_gemini(prompt, fallback)

orders["맞춤메시지"] = orders.apply(msg, axis=1)
print(orders[["고객ID", "세그먼트", "맞춤메시지"]])
